# AGMTL Old-Main Style Training Notebook

This notebook follows the same direct flow as `legacy/Old-Main.ipynb`:

1. Import libraries.
2. Load dataset generators.
3. Define the attention/model.
4. Compile and call `model.fit(...)` directly.
5. Evaluate, save history, and plot results.

Default run: **fold_1, proposed AGMTL-DenseCBAM, artifact_mix**.


In [ ]:
# Import necessary libraries and modules
print("Load Models")

import os
import gc
import math
import time
import random
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")

import tensorflow as tf
from tensorflow.keras import Model, layers, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import Sequence as KerasSequence, to_categorical
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score, classification_report

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
tf.config.optimizer.set_jit(False)
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception as exc:
    print("Mixed precision not enabled:", exc)


In [ ]:
# Settings - edit these before running
img_size = (224, 224)
batch_size = 8
epochs = 50
learning_rate = 1e-4
weight_decay = 1e-2
random_state = 42

# Old-Main style: train one folder split directly.
# Use data_5_fold/fold_1 for one fold, or data if you want the original fixed split.
DATA_ROOT = Path("data_5_fold/fold_1")

# Options:
# MODEL_TYPE = "proposed"  -> DenseNet201 + CBAM + TMD head + artifact head
# MODEL_TYPE = "benchmark" -> DenseNet201 + connected self-attention + TMD head only
MODEL_TYPE = "proposed"

# Options:
# SCENARIO = "clean"        -> no synthetic artifacts
# SCENARIO = "artifact_mix" -> none / motion blur / gaussian noise / metal streak
SCENARIO = "artifact_mix"

TMD_LABELS = ["normal", "subluxation"]
DISPLAY_LABELS = ["Normal", "Subluxation"]
ARTIFACT_LABELS = ["none", "motion_blur", "gaussian_noise", "metal_streak"]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
OUTPUT_DIR = Path(f"oldstyle_results/{MODEL_TYPE}_{SCENARIO}_{DATA_ROOT.name}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT.resolve())
print("MODEL_TYPE:", MODEL_TYPE)
print("SCENARIO:", SCENARIO)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


In [ ]:
# Reproducibility
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

seed_everything(random_state)


In [ ]:
# Load dataset index

def index_split_dataset(root: Path) -> pd.DataFrame:
    rows = []
    class_to_idx = {name: idx for idx, name in enumerate(TMD_LABELS)}
    for split in ["train", "validation", "test"]:
        split_dir = root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"Missing split folder: {split_dir}")
        for class_name in TMD_LABELS:
            class_dir = split_dir / class_name
            if not class_dir.exists():
                raise FileNotFoundError(f"Missing class folder: {class_dir}")
            for path in sorted(class_dir.rglob("*")):
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({
                        "filepath": str(path),
                        "split": split,
                        "class_name": class_name,
                        "tmd_label": class_to_idx[class_name],
                    })
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No images found under {root}")
    return df

df = index_split_dataset(DATA_ROOT)
print(df.groupby(["split", "class_name"]).size())

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("train:", len(train_df), "validation:", len(val_df), "test:", len(test_df))


In [ ]:
# Artifact functions - same concept as the training script

def _ensure_uint8(image: np.ndarray) -> np.ndarray:
    return np.clip(image, 0, 255).astype(np.uint8)


def add_motion_blur(image: np.ndarray, kernel_size: int = 9) -> np.ndarray:
    kernel_size = max(3, int(kernel_size) | 1)
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    kernel[kernel_size // 2, :] = 1.0 / kernel_size
    return _ensure_uint8(cv2.filter2D(image, -1, kernel))


def add_gaussian_noise(image: np.ndarray, sigma: float = 12.0) -> np.ndarray:
    noise = np.random.normal(0, sigma, image.shape)
    return _ensure_uint8(image.astype(np.float32) + noise)


def add_metal_streak(image: np.ndarray, num_streaks: int = 2) -> np.ndarray:
    h, w = image.shape[:2]
    output = image.copy().astype(np.float32)
    for _ in range(max(1, num_streaks)):
        overlay = np.zeros_like(output)
        intensity = np.random.randint(150, 235)
        thickness = np.random.randint(1, 5)
        cv2.line(
            overlay,
            (np.random.randint(0, w), np.random.randint(0, h)),
            (np.random.randint(0, w), np.random.randint(0, h)),
            (intensity, intensity, intensity),
            thickness,
        )
        overlay = cv2.GaussianBlur(overlay, (0, 0), sigmaX=np.random.uniform(2.0, 5.0))
        output = np.maximum(output, overlay)
    return _ensure_uint8(output)


def apply_artifact(image: np.ndarray, artifact_label: int) -> np.ndarray:
    if artifact_label == 0:
        return image
    if artifact_label == 1:
        return add_motion_blur(image, kernel_size=int(np.random.choice([5, 7, 9, 11])))
    if artifact_label == 2:
        return add_gaussian_noise(image, sigma=float(np.random.uniform(8.0, 18.0)))
    if artifact_label == 3:
        return add_metal_streak(image, num_streaks=int(np.random.choice([1, 2, 3])))
    raise ValueError(f"Unknown artifact label: {artifact_label}")


In [ ]:
# Data generator

class TMJSequence(KerasSequence):
    def __init__(self, dataframe, image_size, batch_size, multi_task, scenario, training, seed=42):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_size = image_size
        self.batch_size = batch_size
        self.multi_task = multi_task
        self.scenario = scenario
        self.training = training
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.df) / self.batch_size)

    def on_epoch_end(self):
        if self.training:
            self.rng.shuffle(self.indices)

    def _artifact_for(self, filepath: str) -> int:
        if self.scenario == "clean":
            return 0
        if self.training:
            return int(self.rng.integers(0, len(ARTIFACT_LABELS)))
        digest = hashlib.md5(filepath.encode("utf-8")).hexdigest()
        return int(digest, 16) % len(ARTIFACT_LABELS)

    def _load_image(self, filepath: str) -> np.ndarray:
        image = cv2.imread(filepath)
        if image is None:
            raise ValueError(f"Failed to load image: {filepath}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return cv2.resize(image, self.image_size, interpolation=cv2.INTER_AREA)

    def __getitem__(self, index):
        batch_df = self.df.iloc[self.indices[index * self.batch_size:(index + 1) * self.batch_size]]
        images, tmd_labels, artifact_labels = [], [], []
        for row in batch_df.itertuples(index=False):
            image = self._load_image(row.filepath)
            if self.training and self.rng.random() < 0.5:
                image = cv2.flip(image, 1)
            artifact_label = self._artifact_for(row.filepath)
            image = apply_artifact(image, artifact_label)
            image = image.astype(np.float32) / 255.0
            images.append(image)
            tmd_labels.append(int(row.tmd_label))
            artifact_labels.append(int(artifact_label))
        x = np.stack(images)
        y_tmd = to_categorical(np.array(tmd_labels), num_classes=len(TMD_LABELS))
        y_artifact = to_categorical(np.array(artifact_labels), num_classes=len(ARTIFACT_LABELS))
        if self.multi_task:
            return x, {"tmd_output": y_tmd, "artifact_output": y_artifact}
        return x, y_tmd

multi_task = MODEL_TYPE == "proposed"
train_gen = TMJSequence(train_df, img_size, batch_size, multi_task, SCENARIO, training=True, seed=random_state)
val_gen = TMJSequence(val_df, img_size, batch_size, multi_task, SCENARIO, training=False, seed=random_state)
test_gen = TMJSequence(test_df, img_size, batch_size, multi_task, SCENARIO, training=False, seed=random_state)

print("train batches:", len(train_gen), "validation batches:", len(val_gen), "test batches:", len(test_gen))


In [ ]:
# Define attention blocks and model builders

class AttentionBlock(layers.Layer):
    def __init__(self, attention_type="cbam", reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.attention_type = attention_type.lower()
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        filters = int(input_shape[-1])
        reduced_filters = max(filters // self.reduction_ratio, 1)
        if self.attention_type == "self":
            qk_filters = max(filters // 8, 1)
            self.query_conv = layers.Conv2D(qk_filters, 1, padding="same")
            self.key_conv = layers.Conv2D(qk_filters, 1, padding="same")
            self.value_conv = layers.Conv2D(filters, 1, padding="same")
        elif self.attention_type == "cbam":
            self.avg_pool = layers.GlobalAveragePooling2D()
            self.max_pool = layers.GlobalMaxPooling2D()
            self.shared_dense_1 = layers.Dense(reduced_filters, activation="relu")
            self.shared_dense_2 = layers.Dense(filters)
            self.spatial_conv = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        else:
            raise ValueError(f"Unsupported attention type: {self.attention_type}")
        super().build(input_shape)

    def call(self, inputs):
        if self.attention_type == "self":
            shape = tf.shape(inputs)
            batch_size, height, width = shape[0], shape[1], shape[2]
            channels = inputs.shape[-1]
            q = tf.reshape(self.query_conv(inputs), [batch_size, height * width, -1])
            k = tf.reshape(self.key_conv(inputs), [batch_size, height * width, -1])
            v = tf.reshape(self.value_conv(inputs), [batch_size, height * width, channels])
            attention_scores = tf.matmul(q, k, transpose_b=True)
            attention_scores = attention_scores / tf.math.sqrt(tf.cast(tf.shape(k)[-1], tf.float32))
            attention_weights = tf.nn.softmax(attention_scores, axis=-1)
            attended = tf.matmul(attention_weights, v)
            attended = tf.reshape(attended, [batch_size, height, width, channels])
            return inputs + attended

        avg_descriptor = self.shared_dense_2(self.shared_dense_1(self.avg_pool(inputs)))
        max_descriptor = self.shared_dense_2(self.shared_dense_1(self.max_pool(inputs)))
        channel_attention = tf.nn.sigmoid(avg_descriptor + max_descriptor)
        channel_attention = tf.reshape(channel_attention, (-1, 1, 1, inputs.shape[-1]))
        x = inputs * channel_attention
        avg_map = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_map = tf.reduce_max(x, axis=-1, keepdims=True)
        spatial_attention = self.spatial_conv(tf.concat([avg_map, max_map], axis=-1))
        return x * spatial_attention


def make_backbone():
    backbone = tf.keras.applications.DenseNet201(
        include_top=False,
        weights="imagenet",
        input_shape=(*img_size, 3),
        pooling=None,
    )
    backbone.trainable = True
    return backbone


def build_benchmark_model():
    backbone = make_backbone()
    pool3 = backbone.get_layer("pool3_relu").output
    pool3_att = AttentionBlock("self", name="benchmark_self_attention")(pool3)
    pool3_down = layers.AveragePooling2D(pool_size=2, name="benchmark_pool3_downsample")(pool3_att)
    conv5 = backbone.get_layer("conv5_block32_concat").output
    pool3_proj = layers.Conv2D(int(conv5.shape[-1]), 1, padding="same", name="benchmark_pool3_projection")(pool3_down)
    fused = layers.Concatenate(name="benchmark_fused_features")([conv5, pool3_proj])
    fused = layers.Conv2D(1024, 1, activation="relu", padding="same", name="benchmark_fusion_conv")(fused)
    x = layers.GlobalAveragePooling2D(name="benchmark_gap")(fused)
    x = layers.Dense(512, activation="relu", kernel_regularizer=l2(weight_decay), name="benchmark_fc1")(x)
    x = layers.Dropout(0.5, name="benchmark_dropout")(x)
    x = layers.BatchNormalization(name="benchmark_bn")(x)
    x = layers.Dense(128, activation="relu", name="benchmark_fc2")(x)
    output = layers.Dense(len(TMD_LABELS), activation="softmax", dtype="float32", name="tmd_output")(x)
    return Model(backbone.input, output, name="DenseNet201_Benchmark_SelfAttention")


def build_proposed_model():
    backbone = make_backbone()
    conv5 = backbone.get_layer("conv5_block32_concat").output
    x = AttentionBlock("cbam", name="cbam_attention")(conv5)
    x = layers.Conv2D(1024, 1, activation="relu", padding="same", name="cbam_refine_conv")(x)
    x = layers.GlobalAveragePooling2D(name="shared_gap")(x)
    x = layers.Dense(512, activation="relu", kernel_regularizer=l2(weight_decay), name="shared_fc1")(x)
    x = layers.Dropout(0.5, name="shared_dropout")(x)
    x = layers.BatchNormalization(name="shared_bn")(x)
    x = layers.Dense(128, activation="relu", name="shared_fc2")(x)
    tmd_output = layers.Dense(len(TMD_LABELS), activation="softmax", dtype="float32", name="tmd_output")(x)
    artifact_output = layers.Dense(len(ARTIFACT_LABELS), activation="softmax", dtype="float32", name="artifact_output")(x)
    return Model(backbone.input, [tmd_output, artifact_output], name="AGMTL_DenseCBAM")


In [ ]:
# Build and compile model

if MODEL_TYPE == "benchmark":
    model = build_benchmark_model()
    model.compile(
        optimizer=Adam(learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
        jit_compile=False,
    )
elif MODEL_TYPE == "proposed":
    model = build_proposed_model()
    model.compile(
        optimizer=Adam(learning_rate),
        loss={
            "tmd_output": "categorical_crossentropy",
            "artifact_output": "categorical_crossentropy",
        },
        loss_weights={
            "tmd_output": 1.0,
            "artifact_output": 0.35,
        },
        metrics={
            "tmd_output": ["accuracy"],
            "artifact_output": ["accuracy"],
        },
        jit_compile=False,
    )
else:
    raise ValueError("MODEL_TYPE must be 'benchmark' or 'proposed'")

model.summary()


In [ ]:
# Train the model directly - this is the cell that starts training

checkpoint_path = OUTPUT_DIR / f"{MODEL_TYPE}_{SCENARIO}_{DATA_ROOT.name}_best.keras"
callbacks = [
    ModelCheckpoint(str(checkpoint_path), monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=1),
]

history = model.fit(
    train_gen,
    epochs=epochs,
    validation_data=val_gen,
    verbose=1,
    callbacks=callbacks,
)

print("Training complete.")
print("Best model checkpoint:", checkpoint_path)


In [ ]:
# Evaluate the model on the test set

def collect_predictions(model, generator, multi_task):
    y_true, y_pred, y_conf = [], [], []
    for i in range(len(generator)):
        x, y = generator[i]
        pred = model.predict(x, verbose=0)
        pred_tmd = pred[0] if multi_task else pred
        true_tmd = y["tmd_output"] if multi_task else y
        y_true.extend(np.argmax(true_tmd, axis=1).tolist())
        y_pred.extend(np.argmax(pred_tmd, axis=1).tolist())
        y_conf.extend(np.max(pred_tmd, axis=1).tolist())
    return np.array(y_true), np.array(y_pred), np.array(y_conf)

start = time.perf_counter()
y_true, y_pred, y_conf = collect_predictions(model, test_gen, multi_task)
elapsed = time.perf_counter() - start
images_per_second = len(y_true) / max(elapsed, 1e-8)
latency_ms = 1000.0 / max(images_per_second, 1e-8)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
TN, FP, FN, TP = cm.ravel()
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
specificity = TN / max(TN + FP, 1)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall/Sensitivity: {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Images/sec: {images_per_second:.2f}")
print(f"Latency/image: {latency_ms:.2f} ms")
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=DISPLAY_LABELS, zero_division=0))

results_df = pd.DataFrame([{
    "model_type": MODEL_TYPE,
    "scenario": SCENARIO,
    "fold": DATA_ROOT.name,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "specificity": specificity,
    "f1": f1,
    "tn": int(TN),
    "fp": int(FP),
    "fn": int(FN),
    "tp": int(TP),
    "images_per_second": images_per_second,
    "latency_ms": latency_ms,
    "epochs_ran": len(history.history.get("loss", [])),
}])

results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)
pd.DataFrame(cm, index=DISPLAY_LABELS, columns=DISPLAY_LABELS).to_csv(OUTPUT_DIR / "confusion_matrix.csv")
pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "confidence": y_conf}).to_csv(OUTPUT_DIR / "predictions.csv", index=False)
results_df


In [ ]:
# Save history
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(OUTPUT_DIR / "history.csv", index=False)
hist_df.tail()


In [ ]:
# Plot training history

def plot_history(history, model_name):
    hist = history.history
    loss_key = "loss"
    val_loss_key = "val_loss"
    acc_keys = [k for k in hist.keys() if k.endswith("accuracy") and not k.startswith("val_")]
    val_acc_keys = ["val_" + k for k in acc_keys if "val_" + k in hist]
    epochs_range = np.arange(1, len(hist[loss_key]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, hist[loss_key], label="Train Loss", linewidth=2)
    plt.plot(epochs_range, hist[val_loss_key], label="Validation Loss", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_loss.png", dpi=300)
    plt.show()

    if acc_keys:
        plt.figure(figsize=(10, 5))
        for k in acc_keys:
            plt.plot(epochs_range, hist[k], label=k, linewidth=2)
        for k in val_acc_keys:
            plt.plot(epochs_range, hist[k], label=k, linewidth=2, linestyle="--")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title(f"{model_name} Accuracy")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "training_accuracy.png", dpi=300)
        plt.show()

plot_history(history, f"{MODEL_TYPE}-{SCENARIO}-{DATA_ROOT.name}")


In [ ]:
# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=DISPLAY_LABELS, yticklabels=DISPLAY_LABELS)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {MODEL_TYPE} - {SCENARIO} - {DATA_ROOT.name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=300)
plt.show()
